# CS 195: Natural Language Processing
## Multiclass PyTorch, Adam, and Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F3_3_MulticlassAdamNeuralNetworks.ipynb)


## References

[SLP: Neural Networks, Chapter 6 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/6.pdf)

[Adam vs SGD : What are the optimizers in neural network and when do we use? by Adam Lee](https://medium.com/@pumadd1227/adam-vs-sgd-what-are-the-optimizers-in-neural-network-and-when-do-we-use-238478a0eaea)

[Artificial Neural Networks, Chapter 4 of Machine Learning by Tom M. Mitchell](http://www.cs.cmu.edu/~tom/files/MachineLearningTomMitchell.pdf)

[PyTorch CrossEntropyLoss reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)

[PyTorch Sequential model reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

In [58]:
#import sys
#!{sys.executable} -m pip install datasets sklearn transformers torch

## From last time

Here's the code we used to run the **stochastic gradient descent** training algorithm on our **logistic regression model**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset


data = load_dataset("Deysi/spam-detection-dataset")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

train_labels = [1 if y == "spam" else 0 for y in data["train"]["label"]]
test_labels  = [1 if y == "spam" else 0 for y in data["test"]["label"]]

X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)

y_train = torch.tensor(train_labels, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(test_labels, dtype=torch.float32).unsqueeze(1)

model = torch.nn.Linear(5000, 1) 
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    optimizer.zero_grad() 

    logits = model(X_train) 
    loss = loss_fn(logits, y_train) 

    loss.backward() 
    optimizer.step() 
    
    print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")


with torch.no_grad():
    predictions = torch.sigmoid(model(X_test))
    predicted_labels = (predictions > 0.5).int()
    accuracy = (predicted_labels == y_test.int()).float().mean()

print("Accuracy:", accuracy.item())

## Multiclass Problem Variant

When we want to predict a single label among more than 2 classes, it's called **multiclass** classification.

Let's consider a new multiclass dataset:  https://huggingface.co/datasets/dair-ai/emotion

In [2]:
from datasets import load_dataset
data = load_dataset("dair-ai/emotion")
data

README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Notice that the training labels are already numbers for this one

In [3]:
data["train"][0:5]

{'text': ['i didnt feel humiliated',
  'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake',
  'im grabbing a minute to post i feel greedy wrong',
  'i am ever feeling nostalgic about the fireplace i will know that it is still on the property',
  'i am feeling grouchy'],
 'label': [0, 0, 3, 2, 3]}

In [5]:
data["train"].features

{'text': Value('string'),
 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}

## PyTorch Changes for multiclass

We  need to use the `nn.CrossEntropyLoss()` loss function
* requires the target labels (`y_train` and `y_test`) to be in a 1-dimension tensor of integers rather than a 2-dimensional tensor of floats like `nn.BCEWithLogitsLoss()`
* for evaluation, we only need to find which class had the highest output, so we do an `argmax` on the output logits which returns the index of the largest value in the output tensor


In [59]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]
train_labels = data["train"]["label"]
test_labels = data["test"]["label"]

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)

print(train_labels)

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)

print(y_train)

model = torch.nn.Linear(5000, 6) 
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(1000):
    optimizer.zero_grad() 

    logits = model(X_train) 
    loss = loss_fn(logits, y_train) 

    loss.backward() 
    optimizer.step() 

    if epoch % 100 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")


with torch.no_grad():
    logits = model(X_test)
    predicted_labels = logits.argmax(dim=1) # get index of largest value in each row, dim=1 means apply to each row
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

Column([0, 0, 3, 2, 3, ...])
tensor([0, 0, 3,  ..., 1, 3, 0])
Epoch 1, loss = 1.7934
Epoch 101, loss = 1.5842
Epoch 201, loss = 1.5689
Epoch 301, loss = 1.5622
Epoch 401, loss = 1.5570
Epoch 501, loss = 1.5522
Epoch 601, loss = 1.5477
Epoch 701, loss = 1.5432
Epoch 801, loss = 1.5387
Epoch 901, loss = 1.5343
Accuracy: 0.39750000834465027


What do these predictions look like?

In [61]:
print(logits.argmax(dim=1))

tensor([1, 0, 1,  ..., 1, 1, 1])


We can count how many of each label we have with `bincount`

In [60]:
torch.bincount(predicted_labels)

tensor([ 120, 1880])

### Group Discussion

How good are these results? For reference, here is the dataset: https://huggingface.co/datasets/dair-ai/emotion

Then, rerun the experiment with a different optimization algorithm called **Adam** and discuss whether it made a difference, and if so, by how much


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.1)

## Adam Optimizer

The Adam Optimizer is a stochastic gradient descent variant that includes momentum

Remember the gradient of weight $w_j$ is $g_j = (\hat{y}-y)x_j$

We could calculate the momentum of the gradient as a kind of rolling average of the old gradients and the new gradient

$$m_j = 0.9(\mbox{old } m_j) + 0.1(\mbox{new } g_j)$$

Adam uses this (representing the memory of the gradient *direction*) as well as a squared version, representing the memory of the gradient *magnitude*

$$v_j = 0.9(\mbox{old } v_j) + 0.1(\mbox{new } g_j^2)$$

Of course, you can use any split, not just 0.9 / 0.1, so this is usually written using $\beta$ parameters

$$m_j = \beta_1 m_j + (1-\beta_1)g_j$$


$$v_j = \beta_2 v_j + (1-\beta_2)g_j^2$$

but it doesn't use these as the raw replacement for the weight updates. It first divides them by $(1-\beta^t)$ for time step $t$,


$$\hat{m_j} = m_j/(1-\beta_1^t)$$


$$\hat{v_j} = v_j/(1-\beta_2^t)$$

which inflates the momentum estimates in the beginning but eventually disappears since each $\beta^t$ will get close to 0 as time goes on, $(1-\beta^t)$ will be close to 1.

And the udpate rules is changed from vanilla gradient descent

$$w_j \leftarrow w_j - \eta(\hat{y}-y)x_j$$

to 

$$w_j \leftarrow w_j - \eta\left(\frac{\hat{m_j}}{\sqrt{\hat{v_j}}+\varepsilon}\right)$$


### Why does this work better?

It scales the weight updates on a per-parameter basis
* frequent words get smaller steps
* rare words get larger steps

In SGD, frequent features dominate the gradient - info from rarer words updates slowly

## Let's talk about the new loss function

Our new loss function

In [ ]:
loss_fn = nn.CrossEntropyLoss()

doesn't include the sigmoid activation function, instead it does a **softmax** which rescales the raw output (the *logits*) to all be between 0 and 1, representing probabilities for each output

Example: notice that each value is between 0 and 1 and each row adds up to 1. Details are here: https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html

In [43]:
sm = nn.Softmax(dim=1) # dim = 1 means apply it to each row
example_logits = torch.tensor([[-1.5, 0.3, 3.6],[-1.8,-1.6,-1.4]])
sm(example_logits)

tensor([[0.0058, 0.0354, 0.9588],
        [0.2693, 0.3289, 0.4018]])

## Neural Networks

Hopefully neural networks are familiar to you from your Machine Learning course - but here is a review of some important aspects

<div>
    <img src="images/fullyconnected.png">
</div>

image credit: http://neuralnetworksanddeeplearning.com/chap6.html

For NLP, vectors representing words/sequences-of-words are the input layer (integer encoding, BoW, TF-IDF, etc.)

Output layer: the class for text classification, the next word in the sequence, etc.

## Neural Network Nodes

<div>
    <img src="images/ann-perceptron.png">
</div>

image credit: Machine Learning by Tom M. Mitchell, Chapter 4, http://www.cs.cmu.edu/~tom/files/MachineLearningTomMitchell.pdf

## Activation Functions

The basic perceptron *squashing function* just calls anything positive a 1 and anything negative a 0, but modern neural networks use many other activation functions.


<div>
    <img src="images/activation_binary_step.png" width=300>
</div>

Activation Function images from https://en.wikipedia.org/wiki/Activation_function



### Sigmoid Function

$\sigma (x) = {\frac {1}{1+e^{-x}}}$

<div>
    <img src="images/activation_logistic.png" width=300>
</div>

differentiable, so the calculus in the training algorithm works out

often used in an output layer where you have a binary classification

### Hyperbolic Tangent Function

$\tanh(x) = {\frac {e^{x}-e^{-x}}{e^{x}+e^{-x}}}$

<div>
    <img src="images/activation_tanh.png" width=300>
</div>

like sigmoid, but approximates identity near origin - learns efficiently with small, random, initial weights


### Identity Function

$f(x) = x$

<div>
    <img src="images/activation_identity.png" width=300>
</div>

take the output from the node as is - helpful if your output is numerical

### Rectified-Linear Unit - ReLU

$f(x) = \mbox{max}(0,x)$

<div>
    <img src="images/activation_relu.png" width=300>
</div>

either "doesn't fire" or "fires with measurable intensity" - biologically motivated

often used in hidden layers


### Softmax

Used for outpus layer when you have more than two possible classes - like if you are predicting the next word

Like arg-max (which argument results in the maximum value)

Which class has the largest output value?

However, it is *soft* in that it really applies a probability to every possible value, weighted heavily to the one with the largest value

## Training a neural network: Backpropagation Algorithm

1. Start with random weights or weights learned from some other related task
2. Feed a training example into the network and get a prediction
3. Calculate the **loss function** a measurement of how far away the prediction was from the target value
4. Adjust weights in an attempt to reduce the loss (Calculus)
    * derivative of the loss function with respect to the weights of the network
    * start with the output layer (works just like logistic regression) and move towards the front of the network (backpropagation)
    * adjustments for middle layers based on adjustment of later layer

Visualization: https://youtu.be/Ilg3gGewQ5U?si=3ftp9GcvPt4cKuKY&t=24

## Building a neural network in PyTorch

Very little about our code needs to change logistic regression to a fully-connected neural network

Here's a model which has a hidden layer with 100 nodes that use a ReLU activation function that is fully-connected to the final output layer of 6 nodes

In [62]:
model = nn.Sequential(
    nn.Linear(5000, 100),
    nn.ReLU(),
    nn.Linear(100, 6)
)

## Group Exercise

Change the multiclass code above to use this model instead of the basic one-node linear model. What do you notice?

## Applied Exploration

Select another multiclass classification dataset and get it working with your PyTorch neural network.

Experiment with different numbers of layers, numbers of nodes in each layer, learning rates, and number of epochs. Record your results.

Give a short write-up on the following
* Describe your dataset, including the distribution of the target variable
* Describe the results of the machine learning experiment
* Interpret the results - How did this dataset compare with the emptions dataset? Why do you think you got the results that you did?